In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

# ============================================================================
# LSTM MODEL (WILL ACTUALLY WORK FOR YOUR DATA)
# ============================================================================

class SensorDataset(Dataset):
    def __init__(self, df, context_len=512, forecast_len=96):
        self.context_len = context_len
        self.forecast_len = forecast_len
        self.series_list = []
        
        # Group by series
        for series_id in df['unique_id'].unique():
            series_data = df[df['unique_id'] == series_id].sort_values('ds')['y'].values
            if len(series_data) >= context_len + forecast_len:
                self.series_list.append((series_id, series_data))
    
    def __len__(self):
        total = 0
        for _, series in self.series_list:
            total += len(series) - self.context_len - self.forecast_len + 1
        return total
    
    def __getitem__(self, idx):
        # Find which series
        for series_id, series_data in self.series_list:
            max_starts = len(series_data) - self.context_len - self.forecast_len + 1
            if idx < max_starts:
                start = idx
                context = series_data[start:start + self.context_len]
                target = series_data[start + self.context_len:start + self.context_len + self.forecast_len]
                return (
                    torch.FloatTensor(context).unsqueeze(-1),
                    torch.FloatTensor(target)
                )
            idx -= max_starts
        raise IndexError()

class LSTMForecaster(nn.Module):
    def __init__(self, hidden_size=256, num_layers=3, forecast_len=96, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc1 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc2 = nn.Linear(hidden_size // 2, forecast_len)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # x: [batch, seq_len, 1]
        _, (h_n, _) = self.lstm(x)
        # Use last hidden state
        out = self.relu(self.fc1(h_n[-1]))
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# ============================================================================
# TRAIN LSTM
# ============================================================================

print("=" * 80)
print("TRAINING LSTM")
print("=" * 80)

# Create datasets
train_val_data = pd.concat([train_nhits, valid_nhits], ignore_index=True)
train_dataset = SensorDataset(train_val_data, context_len=512, forecast_len=96)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)

# Model
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = LSTMForecaster(hidden_size=256, num_layers=3, forecast_len=96, dropout=0.3).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Train
epochs = 100
best_loss = float('inf')

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for context, target in train_loader:
        context = context.to(device)
        target = target.to(device)
        
        optimizer.zero_grad()
        pred = model(context)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), 'best_lstm.pt')

print(f"\n✓ Training complete! Best loss: {best_loss:.4f}")

# ============================================================================
# EVALUATE ON TEST DATA
# ============================================================================

model.load_state_dict(torch.load('best_lstm.pt'))
model.eval()

def forecast_series(model, series_data, context_len=512, forecast_len=96):
    """Forecast using last context_len points."""
    context = series_data[-context_len:]
    context_tensor = torch.FloatTensor(context).unsqueeze(0).unsqueeze(-1).to(device)
    
    with torch.no_grad():
        forecast = model(context_tensor)
    
    return forecast.cpu().numpy().flatten()

# Generate forecasts for test data
lstm_forecasts = []

for series_id in test_nhits['unique_id'].unique():
    # Get full series (train + valid)
    full_series = train_val_data[train_val_data['unique_id'] == series_id].sort_values('ds')['y'].values
    
    if len(full_series) >= 512:
        # Forecast
        forecast = forecast_series(model, full_series, context_len=512, forecast_len=96)
        
        # Create forecast dataframe
        for i, val in enumerate(forecast):
            lstm_forecasts.append({
                'unique_id': series_id,
                'ds': i,
                'LSTM': val
            })

lstm_forecast_df = pd.DataFrame(lstm_forecasts)

print(f"\n✓ Generated {len(lstm_forecast_df)} LSTM forecasts")

# ============================================================================
# PLOT LSTM VS ACTUAL
# ============================================================================

# Get actual test values for comparison
test_actuals = []
for series_id in test_nhits['unique_id'].unique():
    test_series = test_nhits[test_nhits['unique_id'] == series_id].sort_values('ds')
    for i, row in test_series.head(96).iterrows():
        test_actuals.append({
            'unique_id': series_id,
            'ds': row['ds'] - test_series['ds'].min(),  # Reset to 0
            'y': row['y']
        })

test_actual_df = pd.DataFrame(test_actuals)

# Merge
lstm_results = lstm_forecast_df.merge(test_actual_df, on=['unique_id', 'ds'], how='left')

# Plot using previous function
plot_cv_results_modified(lstm_results, train_val_data, num_examples=10, pred_col='LSTM')
